## Part A

In [1]:
!pip install --upgrade pandas --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 728.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 31.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.


In [6]:
import pandas as pd

df = pd.read_csv("/content/IMDB top 1000.csv")
df.to_parquet("imdb_raw.parquet", engine="pyarrow")

## Part B

In [7]:
df_clean = pd.read_parquet("imdb_raw.parquet", engine="pyarrow")

# Drop rows where 'Title' or 'Genre' is missing
df_clean = df_clean.dropna(subset=["Title", "Genre"])

# Standardize column names
df_clean.columns = [col.strip().lower().replace(" ", "_") for col in df_clean.columns]

# Convert 'duration' column to numeric (remove 'min')
df_clean["duration"] = df_clean["duration"].str.replace(" min", "", regex=False)
df_clean["duration"] = pd.to_numeric(df_clean["duration"], errors="coerce")

# Save cleaned data
df_clean.to_parquet("imdb_cleaned.parquet", engine="pyarrow")

In [10]:
df_clean.tail(10)

,unnamed:_0,title,certificate,duration,genre,rate,metascore,description,cast,info
990,990,393. 12 Monkeys (1995),R,129,"Mystery, Sci-Fi, Thriller",8.0,74.0,"In a future world devastated by disease, a con...","Director: Terry Gilliam | Stars: Bruce Willis,...","Votes: 571,158 | Gross: $57.14M"
991,991,394. Ghost in the Shell (1995),TV-MA,83,"Animation, Action, Crime",8.0,76.0,A cyborg policewoman and her partner hunt a my...,"Director: Mamoru Oshii | Stars: Atsuko Tanaka,...","Votes: 126,538 | Gross: $0.52M"
992,992,395. The Nightmare Before Christmas (1993),PG,76,"Animation, Family, Fantasy",8.0,82.0,"Jack Skellington, king of Halloween Town, disc...","Director: Henry Selick | Stars: Danny Elfman, ...","Votes: 288,364 | Gross: $75.08M"
993,993,396. Groundhog Day (1993),PG,101,"Comedy, Fantasy, Romance",8.0,72.0,A weatherman finds himself inexplicably living...,"Director: Harold Ramis | Stars: Bill Murray, A...","Votes: 571,642 | Gross: $70.91M"
994,994,"397. Blood In, Blood Out (1993)",R,180,"Crime, Drama",8.0,NaN,Based on the true life experiences of poet Jim...,Director: Taylor Hackford | Stars: Damian Chap...,"Votes: 28,464 | Gross: $4.50M"
995,995,398. Scent of a Woman (1992),R,156,Drama,8.0,NaN,A prep school student needing money agrees to ...,"Director: Martin Brest | Stars: Al Pacino, Chr...","Votes: 256,515 | Gross: $63.90M"
996,996,399. Aladdin (1992),G,90,"Animation, Adventure, Comedy",8.0,86.0,A kindhearted street urchin and a power-hungry...,"Directors: Ron Clements, John Musker | Stars: ...","Votes: 367,489 | Gross: $217.35M"
997,997,400. JFK (1991),R,189,"Drama, History, Thriller",8.0,72.0,New Orleans District Attorney Jim Garrison dis...,"Director: Oliver Stone | Stars: Kevin Costner,...","Votes: 139,634 | Gross: $70.41M"
998,998,301. Nights of Cabiria (1957),Not Rated,110,Drama,8.1,NaN,A waifish prostitute wanders the streets of Ro...,Director: Federico Fellini | Stars: Giulietta ...,"Votes: 42,160 | Gross: $0.75M"
999,999,302. Throne of Blood (1957),Not Rated,110,"Drama, History",8.1,NaN,"A war-hardened general, egged on by his ambiti...",Director: Akira Kurosawa | Stars: Toshirô Mifu...,"Votes: 45,579"


In [9]:
print(df_clean.columns)

Index(['unnamed:_0', 'title', 'certificate', 'duration', 'genre', 'rate',
       'metascore', 'description', 'cast', 'info'],
      dtype='str')


## Part C

In [11]:
df_cleaner = pd.read_parquet("imdb_cleaned.parquet", engine="pyarrow")

In [14]:
# Extract release year from 'title'
df_cleaner["year"] = df_cleaner["title"].str.extract(r"\((\d{4})\)").astype(float)

# Extract vote count from 'info' (pattern: Votes: 1,234,567)
df_cleaner["votes"] = (
    df_cleaner["info"]
    .str.extract(r"Votes:\s*([\d,]+)")
    .replace(",", "", regex=True)
    .astype(float)
)

# Create 'is_high_rated'
df_cleaner["is_high_rated"] = df_cleaner["rate"] >= 8.0

# Filter data
df_filtered = df_cleaner[
    (df_cleaner["year"] > 2000) &
   # (df_cleaner["rate"] >= 8.0) &
    (df_cleaner["votes"] >= 1_000_000)
]

In [15]:
df_filtered.head(10)

,unnamed:_0,title,certificate,duration,genre,rate,metascore,description,cast,info,year,votes,is_high_rated
2,2,3. The Dark Knight (2008),PG-13,152,"Action, Crime, Drama",9.0,84.0,When the menace known as the Joker wreaks havo...,Director: Christopher Nolan | Stars: Christian...,"Votes: 2,260,649 | Gross: $534.86M",2008.0,2260649.0,True
4,4,5. The Lord of the Rings: The Return of the Ki...,PG-13,201,"Action, Adventure, Drama",8.9,94.0,Gandalf and Aragorn lead the World of Men agai...,"Director: Peter Jackson | Stars: Elijah Wood, ...","Votes: 1,614,369 | Gross: $377.85M",2003.0,1614369.0,True
8,8,9. Inception (2010),PG-13,148,"Action, Adventure, Sci-Fi",8.8,74.0,A thief who steals corporate secrets through t...,Director: Christopher Nolan | Stars: Leonardo ...,"Votes: 2,022,655 | Gross: $292.58M",2010.0,2022655.0,True
10,10,11. The Lord of the Rings: The Fellowship of t...,PG-13,178,"Action, Adventure, Drama",8.8,92.0,A meek Hobbit from the Shire and eight compani...,"Director: Peter Jackson | Stars: Elijah Wood, ...","Votes: 1,630,106 | Gross: $315.54M",2001.0,1630106.0,True
14,14,15. The Lord of the Rings: The Two Towers (2002),PG-13,179,"Action, Adventure, Drama",8.7,87.0,While Frodo and Sam edge closer to Mordor with...,"Director: Peter Jackson | Stars: Elijah Wood, ...","Votes: 1,459,119 | Gross: $342.55M",2002.0,1459119.0,True
20,20,21. Interstellar (2014),PG-13,169,"Adventure, Drama, Sci-Fi",8.6,74.0,A team of explorers travel through a wormhole ...,Director: Christopher Nolan | Stars: Matthew M...,"Votes: 1,468,447 | Gross: $188.02M",2014.0,1468447.0,True
35,35,36. The Prestige (2006),PG-13,130,"Drama, Mystery, Sci-Fi",8.5,66.0,"After a tragic accident, two stage magicians e...",Director: Christopher Nolan | Stars: Christian...,"Votes: 1,165,395 | Gross: $53.09M",2006.0,1165395.0,True
36,36,37. The Departed (2006),R,151,"Crime, Drama, Thriller",8.5,85.0,An undercover cop and a mole in the police att...,Director: Martin Scorsese | Stars: Leonardo Di...,"Votes: 1,167,751 | Gross: $132.38M",2006.0,1167751.0,True
61,61,62. Django Unchained (2012),R,165,"Drama, Western",8.4,81.0,"With the help of a German bounty hunter, a fre...",Director: Quentin Tarantino | Stars: Jamie Fox...,"Votes: 1,328,656 | Gross: $162.81M",2012.0,1328656.0,True
62,62,63. The Dark Knight Rises (2012),PG-13,164,"Action, Adventure",8.4,78.0,Eight years after the Joker's reign of anarchy...,Director: Christopher Nolan | Stars: Christian...,"Votes: 1,491,281 | Gross: $448.14M",2012.0,1491281.0,True


In [16]:
df_filtered.shape

(36, 13)

## Part D

In [17]:
df_filtered.to_parquet("imdb_final.parquet", engine="pyarrow")

# Part 2 - Grading Key for The Four Vs of Big Data

**Question:**  
- Considering the 4 Vs of Big Data — Volume, Velocity, Variety, and Veracity:  
- Discuss the challenges posed by each V and strategies for addressing them.
- Part b and c are very subjective (no answer key and no wrong answer)

## Expected Key Points

### 1. Volume  
**Challenge:**  
- Massive scale of data generation (e.g., social media, IoT, online transactions)  
- Traditional systems struggle with storage, retrieval, and processing at this scale  

**Strategies:**  
- Use scalable storage solutions (cloud storage, distributed databases)  
- Implement distributed processing frameworks (e.g., Hadoop, Spark)  

### 2. Velocity  
**Challenge:**  
- Speed of data generation and need for near real-time processing  
- Handling streaming data (financial markets, sensors, web traffic) efficiently  

**Strategies:**  
- Deploy stream processing frameworks (e.g., Apache Kafka, Spark Streaming)  
- Optimize system architecture for low-latency ingestion and analysis  

### 3. Variety  
**Challenge:**  
- Data comes in many formats: structured, semi-structured, and unstructured (e.g., text, video, logs, social media posts)  
- Integrating and analyzing heterogeneous data types  

**Strategies:**  
- Use flexible data models (e.g., NoSQL databases)  
- Build data lakes or multimodal databases to ingest diverse formats  
- Invest in strong data integration and data wrangling tools  

### 4. Veracity  
**Challenge:**  
- Data reliability, accuracy, and trustworthiness are often uncertain  
- Noise, inconsistencies, biases, and errors can affect analysis and decisions  

**Strategies:**  
- Apply rigorous data cleansing, validation, and anomaly detection techniques  
- Prioritize high-veracity data sources and establish data governance policies  